# 08 — NCCS Historical Data Integration (2010–2024)

## Objective
Evaluate whether historical NCCS Form 990 e-file data can expand our significant diversion of assets disclosure modeling dataset beyond the original IRS XML-derived 2023–2025 records.

**Scope:** Tax years 2010–2024, subject to NCCS data availability and comparable field definitions. The target is a *self-reported significant diversion disclosure*, not confirmed fraud.

**Prior validation:** Notebook 07 compared 2023 NCCS and IRS records: 315/315 matched positive disclosures agreed; 12 financial/governance fields exceeded 99.98% exact agreement where both values were observed; Governance joined to all Header records and Summary to 99.71%.

**Workflow:**
1. Inventory historical files already downloaded (run now).
2. Verify historical download availability and obtain missing years (next step, after reviewing output).
3. Measure annual filing counts, unique EINs, and observed positive/negative/missing diversion responses.
4. Assess missingness, filing types, tax-year definitions, and amended filings before creating one record per EIN-year.
5. Produce a documented historical integration dataset for later feature engineering.

Do not classify missing diversion responses as negative. Preserve source filing identifiers and avoid assuming an EIN/year uniquely identifies a filing.


## Step 1 — Inventory locally available NCCS data

Run this cell from the same project used for Notebook 07. It searches upward from the current working directory for `data/external/nccs/efile_v3_1` and reports available files by tax year and table. **This step does not download or modify any data.**


In [ ]:
from pathlib import Path
import pandas as pd

TARGET_YEARS = list(range(2010, 2025))
TABLE_PATTERNS = {
    "header": "F9-P00-T00-HEADER",
    "summary": "F9-P01-T00-SUMMARY",
    "governance": "F9-P06-T00-GOVERNANCE",
}

# Find the existing project data directory without assuming notebook execution location.
candidates = [
    base / "data" / "external" / "nccs" / "efile_v3_1"
    for base in [Path.cwd(), *Path.cwd().parents]
]
existing = [p for p in candidates if p.is_dir()]
if not existing:
    raise FileNotFoundError(
        "Could not locate data/external/nccs/efile_v3_1 from the current "
        f"working directory: {Path.cwd()}. Open Notebook 08 from the project "
        "or set NCCS_DIR explicitly to your existing NCCS folder."
    )
NCCS_DIR = existing[0]
print("Current working directory:", Path.cwd())
print("NCCS data directory:", NCCS_DIR.resolve())

rows = []
for year in TARGET_YEARS:
    year_dir = NCCS_DIR / str(year)
    files = list(year_dir.glob("*.parquet")) if year_dir.is_dir() else []
    files += list(year_dir.glob("*.csv")) if year_dir.is_dir() else []
    files += list(year_dir.glob("*.csv.gz")) if year_dir.is_dir() else []
    for table, prefix in TABLE_PATTERNS.items():
        matching = [p for p in files if p.name.upper().startswith(prefix)]
        rows.append({
            "tax_year_folder": year,
            "table": table,
            "local_files": len(matching),
            "file_names": "; ".join(sorted(p.name for p in matching)),
        })

inventory = pd.DataFrame(rows)
summary = inventory.pivot(index="tax_year_folder", columns="table", values="local_files").fillna(0).astype(int)
summary["all_three_tables_present"] = (summary[list(TABLE_PATTERNS)] > 0).all(axis=1)
print("\nLocal file inventory (counts, not filing records):")
display(summary)
print("\nYears with all three required tables:", summary.index[summary["all_three_tables_present"]].tolist())
print("Years needing at least one table:", summary.index[~summary["all_three_tables_present"]].tolist())
print("\nMatching local files:")
display(inventory[inventory["local_files"] > 0].reset_index(drop=True))


In [ ]:
from pathlib import Path
import requests
import pyarrow.parquet as pq

BASE_URL = (
    "https://nccs-efile.s3.us-east-1.amazonaws.com/"
    "public/efile_v3_1"
)

PROJECT_ROOT = Path.cwd().parent
NCCS_DIR = PROJECT_ROOT / "data/external/nccs/efile_v3_1"

YEARS = range(2010, 2025)

TABLES = {
    "header": "F9-P00-T00-HEADER",
    "summary": "F9-P01-T00-SUMMARY",
    "governance": "F9-P06-T00-GOVERNANCE"
}

downloaded = []
skipped = []
failed = []

for year in YEARS:
    year_dir = NCCS_DIR / str(year)
    year_dir.mkdir(parents=True, exist_ok=True)

    for table_name, prefix in TABLES.items():
        filename = f"{prefix}-{year}.parquet"
        destination = year_dir / filename
        url = f"{BASE_URL}/{filename}"

        if destination.exists():
            try:
                pq.ParquetFile(destination)
                print(f"SKIP: {filename}")
                skipped.append(filename)
                continue
            except Exception:
                print(f"INVALID EXISTING FILE: {filename}")
                destination.unlink()

        print(f"DOWNLOADING: {filename}")

        temporary_file = destination.with_suffix(".parquet.part")

        try:
            with requests.get(
                url,
                stream=True,
                timeout=(30, 180)
            ) as response:
                response.raise_for_status()

                with open(temporary_file, "wb") as f:
                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            f.write(chunk)

            # Verify Parquet structure before accepting the file.
            parquet_file = pq.ParquetFile(temporary_file)
            row_count = parquet_file.metadata.num_rows

            temporary_file.replace(destination)

            downloaded.append(filename)
            print(f"COMPLETE: {filename} ({row_count:,} rows)")

        except Exception as e:
            failed.append((filename, str(e)))
            print(f"FAILED: {filename} - {e}")

            if temporary_file.exists():
                temporary_file.unlink()

print("\nDOWNLOAD SUMMARY")
print("Downloaded:", len(downloaded))
print("Skipped existing:", len(skipped))
print("Failed:", len(failed))
print("Total available:", len(downloaded) + len(skipped))

if failed:
    print("\nFAILED FILES:")
    for filename, error in failed:
        print(filename, error)

### After Step 1

Share the printed path, annual inventory table, and years-with-all-three-tables list. We will use those results to determine the smallest safe download step. Historical availability must be checked rather than assumed from the 2023 sample.
